

# Master Thesis - Computational Pipeline README

Documentation of the computational pipeline and its traceability to the master thesis.

---

## 1. Pipeline overview

```text
01 Data preparation
├──→ 02 EDA                         [diagnostic / design]
└──→ 03 Panel construction
      ├──→ 04 LightGBM training
      │     └──→ 05a Score generation
      │           └──→ 05b CP calibration
      │                 └──→ 06 Coverage evaluation
      │                       └──→ 07 Coverage analysis
      │                             └──→ 08 Local reliability audit
      │                                   └──→ 09 Operational translation
      │
      └──→ 10 Logistic robustness benchmark
             ↑ fits its alternative scorer from the NB03 panel
             ↑ reads primary reference outputs from NB04–NB06 and NB09
```

**Dependency-complete order:** `01 → 03 → 04 → 05a → 05b → 06 → 07 → 08 → 09 → 10`, with `02` as a diagnostic/design notebook. NB10 fits its alternative scorer from the NB03 panel but uses primary-pipeline reference outputs through NB09 for its robustness comparisons.

**Legacy NB07 bridge:** current NB07 writes `results/nb07/nb07_manifest.json`. The executed NB08–NB10 notebooks retain references to the legacy `manifests/analysis_manifest.json`. In the thesis run, these two manifest files are byte-for-byte identical and contain the same NB06 upstream paths. NB08 consumes those bridge paths, NB09 checks the legacy manifest's presence, and NB10 retains it as a legacy/fallback dependency.

---


## 2. Repository layout

```text
master_thesis/
├── code/                     # Notebooks 01–10
├── data/
│   ├── historical_data_*.zip # Freddie Mac source archives consumed by NB01
│   ├── parquet/              # Normalized origination/performance data
│   ├── precomp/              # Precomputed origination, label, prepayment, lag features
│   ├── panel/                # Final split-labelled modelling panel
│   └── panel_temp/           # Intermediate panel material
├── models/
│   ├── lgbm_model.txt
│   └── cat_encoders.json
├── cp_artifacts/
│   ├── isotonic_calibrator.pkl
│   ├── isotonic_calibrator_meta.json
│   ├── aci_cal_scores.npy
│   └── aci_init_state.json
├── manifests/
│   ├── ingestion_manifest.json
│   ├── panel_manifest.json
│   ├── model_manifest.json
│   ├── score_manifest.json
│   ├── cp_manifest.json
│   ├── coverage_manifest.json
│   ├── analysis_manifest.json     # Legacy NB07 bridge referenced by executed NB08–NB10
│   └── audit_manifest.json
└── results/
    ├── scores/                
    ├── nb05a/
    ├── nb05b/
    ├── nb06/
    ├── nb07/
    ├── nb08/
    ├── nb09/
    └── nb10/
```

---


## 3. Notebooks with their inputs and outputs
| Notebook | Role | Main inputs | Main outputs |
|---|---|---|---|
| `01_data_preparation.ipynb` | Reproducible ingestion and normalization of the Freddie Mac release | Freddie Mac historical ZIP archives (`data/historical_data_*.zip`) | `data/parquet/{origination,performance,performance_by_period}/*`; `manifests/ingestion_manifest.json` |
| `02_eda.ipynb` | Source-data validation and exploratory diagnostics informing panel/feature design | Normalized origination/performance data | Diagnostic tables/plots; no downstream file dependency |
| `03_panel_construction.ipynb` | Defines the observation unit, forward label, features, chronological windows, subgroup cells, and final panel | Normalized parquet data + ingestion manifest | `data/precomp/*`; `data/panel/*`; `manifests/panel_manifest.json` |
| `04_lgbm_training.ipynb` | Tunes, fits, evaluates, and interprets the primary LightGBM scorer | Final panel + panel manifest | `models/lgbm_model.txt`; `models/cat_encoders.json`; `manifests/model_manifest.json` |
| `05a_score_generation.ipynb` | Fits and validates isotonic calibration, scores all windows, and computes score-drift diagnostics | Panel + primary model artifacts | `cp_artifacts/isotonic_calibrator*`; `results/scores/*`; `manifests/score_manifest.json` |
| `05b_cp_calibration.ipynb` | Calibrates SCP/Mondrian/APS thresholds and initializes ACI/DtACI fixed-reference states | Calibration scores + score/panel manifests + isotonic-calibrator metadata | `manifests/cp_manifest.json`; `cp_artifacts/{aci_cal_scores.npy,aci_init_state.json}` |
| `06_coverage_evaluation.ipynb` | Evaluates static and monthly adaptive CP, including temporal and prepayment-sensitivity analyses | Score/CP/panel manifests + ACI state/calibration array + scored windows + prepayment lookup | `results/nb06/*`; `manifests/coverage_manifest.json` |
| `07_coverage_analysis.ipynb` | Converts NB06 outputs into thesis-facing reliability, regime, monitoring, and temporal analyses | NB06 outputs + CP/panel manifests | `results/nb07/*`; `results/nb07/nb07_manifest.json` |
| `08_local_reliability_audit.ipynb` | Audits score-decile, subgroup, pocket-level, and miscoverage-concentration reliability | Scored test windows + CP manifest + legacy NB07 bridge | `results/nb08/*`; `manifests/audit_manifest.json` |
| `09_operational_translation.ipynb` | Maps static prediction sets to actions and evaluates workload, capture, matched-budget performance, subgroup burden, and cost sensitivity | Scored test windows + CP/audit manifests + legacy NB07 presence check | `results/nb09/*` |
| `10_logit_robustness_benchmark.ipynb` | Re-runs static-method robustness with a separate penalized-logit scorer and isotonic calibrator | Final panel + primary reference artifacts from NB04–NB06 and NB09 + legacy NB07 bridge | Benchmark discrimination, calibration, coverage, drift, and operational outputs in `results/nb10/` |

---


## 4. Notebook structure in detail

##### `01_data_preparation.ipynb`

| Notebook section | Contents |
|---|---|
| §0 | Environment, paths, release metadata, reproducibility setup |
| §§1–4 | Origination/performance schemas, source-file patterns, SQL projections |
| §§5–11 | DuckDB/ZIP/TXT→Parquet utilities, source-manifest construction, ingestion execution |
| §§12–14 | Output verification, row-count reconciliation, schema checks |
| §15 | Repartition performance data by reporting year |
| §16 | Final validation and ingestion manifest |

##### `02_eda.ipynb`

| Notebook section | Contents |
|---|---|
| §§0–4 | Target semantics, structural checks, origination/performance diagnostics, vintage analysis |
| §5.1 | Monthly positive-class dynamics |
| §5.2 | Vintage delinquency curves |
| §5.3 | COVID/forbearance diagnostics |
| §§5.4–5.5 | Calibration-cell support and calibration-window checks |
| §§5.6–5.7 | Transition-risk class imbalance and geographic-coverage diagnostics |
| §5.8 | Subgroup event rates by regime |
| §5.9 | Feature-level PSI diagnostics |
| §§5.10–5.11 | Risk-bucket monotonicity and feature-availability checks |
| §5.12 | Code-01 termination incidence |

##### `03_panel_construction.ipynb`

| Notebook section | Contents |
|---|---|
| §0 | Split definitions, target horizon, eligible feature set, constants |
| §1 | Census/vintage lookup tables |
| §2 | Base-population definition and diagnostics |
| §3.1 | Origination-feature precompute |
| §3.2 | Forward delinquency-label precompute |
| §3.2b | Forward prepayment lookup |
| §3.3 | Lagged/dynamic-feature precompute |
| §§3.4–3.6 | Precompute summary, right truncation, year-wise panel assembly |
| §4 | Chronological split labels and borrower-risk subgroups |
| §5 | Ten QC checks: rates/trends, contamination, subgroup behavior/support, leakage, structure, reductions, duplicates, label gaps |
| §6 | `panel_manifest.json` |

##### `04_lgbm_training.ipynb`

| Notebook section | Contents |
|---|---|
| §§0–2 | Manifest loading, feature contract, data loading, imbalance handling |
| §3 | Hyperparameter search |
| §4 | Full-scale fit, early stopping, model/encoder persistence |
| §§5.1–5.4 | Window discrimination, subgroup performance, raw-score reliability diagnostics |
| §5.5 | SHAP feature attribution |
| §6 | Model artifact registry / manifest |

##### `05a_score_generation.ipynb`

| Notebook section | Contents |
|---|---|
| §0 | Load panel/model contracts |
| §§1.0–1.1 | Build isotonic-fit sample and fit isotonic calibrator |
| §1.3 | Out-of-sample calibration validation on the disjoint conformal partition |
| §2 | Generate calibrated score files for calibration/test windows |
| §3 | Score-distribution and score-PSI diagnostics |
| §4 | `score_manifest.json` |

##### `05b_cp_calibration.ipynb`

| Notebook section | Contents |
|---|---|
| §§0–1 | Load contracts / conformal-calibration scores and quantile helper |
| §2 | Global split-conformal threshold and score-distribution diagnostic |
| §3 | Mondrian thresholds: crossed FICO×LTV plus coarser partitions |
| §4 | Randomized APS threshold |
| §5 | ACI/DtACI initialization |
| §6 | Calibration/unit checks |
| §7 | Calibration-anchor failure-mode diagnostics: class/subgroup, set-size, threshold spread |
| §8 | `cp_manifest.json` |

##### `06_coverage_evaluation.ipynb`

| Notebook section | Contents |
|---|---|
| §0 | Prediction-set logic and artifact contracts |
| §1 | Calibration-anchor sanity checks |
| §§2–3 | Static-method evaluation and aggregation |
| §4 | Regime and transition diagnostics |
| §5 | Monthly ACI/DtACI evaluation |
| §6 | Monthly, rolling, cumulative coverage and adaptive trajectories |
| §7 | Interval reporting and practical-deviation convention |
| §8 | Persist coverage outputs |
| §8b | Prepayment-restricted sensitivity analysis |
| §9 | Validation and `coverage_manifest.json` |

##### `07_coverage_analysis.ipynb`

| Notebook section | Contents |
|---|---|
| §§0–3 | Registry/load layer and master metric frame, including calibration anchor |
| §4 | Regime-level reliability summaries |
| §5 | Class-conditional analysis |
| §§6–7 | Primary and secondary subgroup views |
| §8 | Prediction-set composition |
| §9 | Temporal/rolling coverage, adaptive trajectories, crisis-window thesis figure |
| §10 | PSI dissociation, mechanism attribution, regime evidence, transition diagnostics |
| §11 | Exploratory static trade-off view |
| §12 | Statistical reporting |
| §13 | Output verification and NB07 bridge manifest |

##### `08_local_reliability_audit.ipynb`

| Notebook section | Contents |
|---|---|
| §§0–1 | Contracts/NB07 bridge, audit sample, reconstruction of calibrated probabilities/miscoverage |
| §2 | Score-decile and predefined-group coverage baselines |
| §3 | Score-only vs full-feature miscoverage prediction |
| §4 | Full reliability model, out-of-sample concentration, feature-importance diagnostics |
| §5 | Interpretable depth-limited reliability pockets |
| §6 | Pocket overlap with predefined cells/regimes |
| §7 | Method synthesis, risk-decile, APS score-band, and pocket diagnostics |
| §8 | Locked outputs and `audit_manifest.json` |

##### `09_operational_translation.ipynb`

| Notebook section | Contents |
|---|---|
| §§0–1 | Load contracts and build deterministic routing dataset |
| §2 | Three-way routing tables and core metric construction |
| §3 | False auto-clear/capture and review-queue quality |
| §4 | Queue contraction and absolute-volume accounting |
| §5 | Probability-ranking capture frontier |
| §6 | Matched non-auto-clear-budget comparison |
| §7 | Borrower-risk-cell routing burden and targeting quality |
| §8 | Monthly non-auto-clear workload stability |
| §9 | Exploratory cost sensitivity |
| §10 | Locked operational findings |

##### `10_logit_robustness_benchmark.ipynb`

| Notebook section | Contents |
|---|---|
| §§0–0.5 | Setup/contracts and primary LightGBM/CP/coverage/PSI/NB09 reference loading |
| §1 | Penalized logistic model, preprocessing, discrimination, coefficients |
| §2 | Raw-score calibration diagnostics and separate isotonic calibrator |
| §3 | Separate SCP/Mondrian/APS threshold calibration |
| §4 | Static-method coverage evaluation |
| §5 | Primary-vs-logistic discrimination, reliability, abstention, and drift |
| §6 | Raw-vs-isotonic CP and score comparison |
| §7 | Operational routing and matched-budget benchmark |
| §8 | Benchmark synthesis |

---


## 5. Thesis subsection: notebook traceability

| Thesis subsection | Computational source / role |
|---|---|
| **1.1 Motivation** | Conceptual/literature. Empirical anchors for rare-event rates and repeated monthly panel structure: NB03 §§3.6, 5. |
| **1.2 Research Questions** | Authorial framing; not generated by a notebook. |
| **1.3 Approach and Thesis Map** | Pipeline design summarized from NB03–NB10. |
| **2.1 Conformal Prediction: Foundations and Marginal Coverage** | Conceptual/theoretical. Split-CP calibration logic: NB05b §§1–2. |
| **2.2 Conditional Coverage and Its Impossibility** | Conceptual/theoretical. Mondrian and APS implementations: NB05b §§3–4. |
| **2.3 Mortgage Panels as Multi-Layered Non-Exchangeability** | Conceptual synthesis plus panel evidence: NB02 §§5.3, 5.9; NB03 §§3.2, 3.6, 4–5. |
| **2.4 The Two-Level Failure Taxonomy** | Authorial taxonomy; empirical anchors: NB05b §§2, 7, NB06 §§4, 6, NB07 §§5–6, 9–10, NB08 §§2–7, NB10 §6. |
| **2.5 The Monitoring Gap: PSI and Conformal Reliability** | Conceptual framing; score PSI: NB05a §3; SCP threshold: NB05b §2; regime/coverage comparison: NB06 §4 and NB07 §10; feature PSI: NB02 §5.9. |
| **2.6 The Operational Question: Prediction Sets as Screening Decisions** | Conceptual routing identity; implemented/tested in NB09 §§1, 5–6. |
| **3.1 Dataset and Observation Unit** | NB01 release/ingestion; NB03 §§0, 2, 3.2, 5 for population/label checks; sampling/evaluation contracts across NB02, NB04–NB06, NB08–NB10. |
| **3.2 Chronological Split and Regime Design** | NB03 §§0, 4–5; regime-supporting diagnostics: NB02 §5.3, NB06 §4, NB07 §10. |
| **3.3 Features and Panel Construction** | NB03 §§1, 3.1, 3.3, 3.6, 4–5; feature-quality/design checks in NB02 §2 and §§5.10–5.11. |
| **3.4 Base Model, Probability Calibration, and Robustness Benchmark** | LightGBM: NB04 §§2–6; isotonic calibration: NB05a §1; logistic benchmark: NB10 §§1–2, 5. |
| **3.5 Conformal Prediction Methods** | Score generation: NB05a §2; threshold/state calibration: NB05b §§1–5; monthly ACI/DtACI: NB06 §5. |
| **3.6 Evaluation Framework** | Coverage/temporal/inference: NB06 §§0–7; PSI: NB05a §3 and NB02 §5.9; local audit: NB08 §§2–7; routing: NB09 §§1–8. |
| **4.1 Marginal Validity and the Failure Taxonomy (RQ1)** | Calibration anchor/score geometry: NB05a §3 and NB05b §§2, 7; regime/temporal coverage: NB06 §§1–4, 6 and NB07 §§4–6, 8–10; pockets: NB08 §§2–7; robustness: NB06 §8b and NB10 §§5–6. |
| **4.2 What Each Conformal Variant Repairs (RQ2)** | Static/adaptive comparison: NB06 §§2–6 and NB07 §§4–9; score-decile/pocket audit: NB08 §§2–7; robustness: NB06 §8b and NB10 §§4–5. |
| **4.3 What a Standard Monitoring Process Sees (RQ3)** | Calibration anchor: NB05b §7; score PSI: NB05a §3; regime/transition and rolling coverage: NB06 §§4, 6 and NB07 §§9–10; feature PSI: NB02 §5.9; benchmark: NB10 §5. |
| **4.4 Operational Translation and the Selection-Efficiency Null (RQ4)** | NB09 §§1–8; logistic robustness in NB10 §7. |
| **5.1 The Taxonomy as an Explanatory Lens** | No new measurement; synthesis of evidence mapped to §§4.1–4.4. |
| **5.2 What Conformal Prediction Contributes When Selection Value Is Null** | No new measurement; interpretation of evidence mapped to §§4.1–4.2 and 4.4. |
| **5.3 Implications for Model Risk Management** | Proposal, not a new experiment; empirical premise from §§4.1, 4.3–4.4. |
| **5.4 Bounds on This Chapter’s Claims** | Authorial limitations; no new notebook analysis. |
| **6.1 Answers to the Research Questions** | Synthesis of Chapter 4; no new computation. |
| **6.2 Contributions and Their Epistemic Status** | Synthesis of Chapters 4–5; no new computation. |
| **6.3 Future Research** | Proposed extensions; no new computation. |

---


## 6. Thesis tables and figures → notebook source

### Main-text tables and figures

| Thesis item | Source notebook(s) | Relevant notebook section / artifact |
|---|---|---|
| **Table 3.1 - Chronological split and regime design** | NB03 | §0 split definitions; §§5–6 validation/summary; `panel_manifest.json` |
| **Table 4.1 - Split conformal reliability by regime** | NB05a + NB05b + NB06 | NB05b §7 calibration anchor; NB06 §§3–4 test metrics/mechanisms; NB05a §3 score PSI |
| **Table 4.2 - Method × regime reliability** | NB06 | §§3, 5 |
| **Table 4.3 - Operational routing summary** | NB09 | §§1–2 and §6; `nb09_routing_metrics.csv`, `nb09_equal_rate_comparison.csv` |
| **Figure 4.1 - SCP nonconformity-score mass by true class** | NB05b | §2; `results/nb05b/figures/fig5_1_scp_score_distribution.*` |
| **Figure 4.2 - Coverage across calibrated-probability deciles** | NB08 | §2; `results/nb08/figures/fig1_score_decile_coverage.*` |
| **Figure 4.3 - Monthly and rolling crisis coverage** | NB06 + NB07 | NB06 §6 temporal series; NB07 §9 thesis-facing plot; `results/nb07/figures/fig5_3_crisis_temporal_coverage.*` |
| **Figure 4.4 - Probability-ranking capture curves / conformal operating points** | NB09 | §5; `results/nb09/figures/fig5_4_capture_budget_frontier.*` |

### Appendix traceability

| Thesis appendix section | Computational source |
|---|---|
| **A.1 Finite-Sample Validity of Split CP** | The theorem/proof discussion is literature-based; empirical threshold/tie diagnostics come from NB05b §§2, 4, 6. |
| **B.1 Data Release, Product Scope, and Panel Construction** | NB01 §§0, 9–16; NB03 §§2–5. |
| **B.2 Label Construction and Measurement Diagnostics** | NB03 §3.2 and §5 boundary/label-gap diagnostics. |
| **B.3 Sample Allocation and Deterministic Hashing** | Cross-notebook synthesis of sampling contracts in NB02, NB03, NB04, NB05a, NB08/NB09, and NB10. |
| **B.4 Feature Dictionary, Exclusions, and Leakage Screen** | NB03 §§0–1, 3.1, 3.3, 3.6, 5; supporting field diagnostics in NB02 §§2, 5. |
| **B.5 Borrower-Risk Cells: Sizes and Event Rates** | NB03 §§4–5. |
| **C.1 LightGBM Configuration, Tuning, and Feature Attribution** | NB04 §§2–5, especially §§3–4 and §5.5. |
| **C.2 Probability-Calibration Diagnostics** | Raw-score diagnostics: NB04 §5.4; isotonic fit/map and out-of-sample validation: NB05a §1. |
| **C.3 Cross-Model Discrimination and Logistic Benchmark Specification** | NB04 §5.1 plus NB10 §§1–2 and §5. |
| **D.1 Calibration Thresholds and Mondrian Partitions** | NB05b §§1–4. |
| **D.2 Adaptive Conformal Configuration** | NB05b §5 and NB06 §5. |
| **D.3 Metric Definitions and Inference Conventions** | Coverage/rolling/interval rules: NB06 §§3, 6–7; PSI: NB05a §3; local reliability: NB08 §§2–5; matched budget: NB09 §§5–6. |
| **D.4 Robustness Design Matrix** | Cross-notebook scope summary: primary pipeline, NB06 §8b, and NB10 §§4–7. |
| **E.1 Complete Static-Method Coverage Results** | NB05b §7; NB06 §§2–3; NB07 §8. |
| **E.2 Effect of Probability Calibration on Class-Conditional Coverage** | NB10 §6. |
| **E.3 Prepayment-Restricted Sensitivity Analysis** | NB06 §8b. |
| **E.4 Alternative Mondrian Partition Results** | NB05b §3 thresholds; NB06 §§2–3 evaluation; NB07 §7 presentation. |
| **E.5 Coverage by Score Decile** | NB08 §2. |
| **E.6 Miscoverage-Prediction Models** | NB08 §3. |
| **E.7 Miscoverage-Concentration Analysis** | NB08 §4. |
| **E.8 Local Reliability Pockets** | NB08 §5. |
| **E.9 Cell-Level Miscoverage Profiles** | NB05b §7; NB08 §§2, 7. |
| **E.10 Adaptive Coverage and Rolling Reliability** | NB06 §§5–6; NB07 §9. |
| **E.11 Static-Method Robustness Across Scorers and Samples** | NB10 §§4–5. |
| **F.1 Score-Distribution Diagnostics by Regime** | NB05a §3; NB06 §4; NB07 §10. |
| **F.2 Regime Classification and Supporting Evidence** | NB07 §10, informed by NB02/NB03 regime diagnostics. |
| **F.3 Feature-Level Population Stability Indices** | NB02 §5.9. |
| **F.4 PSI and Coverage Under the Logistic Benchmark** | NB10 §§4.4, 5.4. |
| **F.5 Monitoring Threshold Classification** | NB05a §3; NB06 §§3–4; NB10 §§4.4–5.4. |
| **G.1 Routing Outcomes by Borrower-Risk Cell** | NB09 §7. |
| **G.2 Monthly Review-Workload Stability** | NB09 §8. |
| **G.3 Absolute Review and Capture Volumes** | NB09 §4. |
| **G.4 Matched-Budget Capture Results** | NB09 §§5–6. |
| **G.5 Operational Results Under the Logistic Benchmark** | NB10 §7. |
| **G.6 Prediction-Set Composition and Routing Semantics** | NB09 §§1–3; NB06 §3 for the randomized-set comparison. |

### Appendix table-level lineage

| Thesis table | Source notebook(s) / section |
|---|---|
| **Table A.1 - Score granularity and realised coverage on the conformal calibration partition** | NB05b §§2, 4, 6. |
| **Table B.1 - From the source files to the analysis panel** | NB01 §§0, 9–16 plus NB03 §§2–5. |
| **Table B.2 - Label and window measurement diagnostics** | NB03 §3.2 and §5 boundary/label-gap checks. |
| **Table B.3 - Allocation of rows across analytical samples** | Cross-notebook synthesis of hash/sample definitions in NB02, NB03, NB04, NB05a, NB08/NB09, and NB10. |
| **Table B.4 - Feature dictionary** | NB03 §§0–1, 3.1, 3.3, 3.6 and §5 feature/schema checks. |
| **Table B.5 - Fields withheld from the feature set** | NB03 §§0, 3.1, 3.6, 5; supporting field diagnostics in NB02 §§2, 5. |
| **Table B.6 - Borrower-risk cells on the calibration window** | NB03 §§4–5. |
| **Table C.1 - LightGBM specification as fitted** | NB04 §§3–4. |
| **Table C.2 - Leading SHAP feature attributions** | NB04 §5.5. |
| **Table C.3 - Raw-score reliability on the calibration window** | NB04 §5.4. |
| **Table C.4 - Isotonic map lookup** | NB05a §1.1. |
| **Table C.5 - Discrimination by window, both scorers** | Primary scorer: NB04 §5.1; logistic scorer/comparison: NB10 §§1, 5.1. |
| **Table D.1 - Conformal thresholds on the calibration window** | NB05b §§2–4. |
| **Table D.2 - Adaptive conformal settings** | NB05b §5 plus the deployed monthly update implementation in NB06 §5. |
| **Table D.3 - Asymptotic no-shift reference for PSI** | NB05a §3. |
| **Table D.4 - Clopper–Pearson intervals for marginal coverage** | NB06 §7; thesis-facing export/check in NB07 §12. |
| **Table D.5 - Robustness design matrix** | Scope synthesis across the primary pipeline, NB06 §8b, and NB10 §§4–7. |
| **Table E.1 - Static-method coverage / set composition** | NB06 §§2–3 and set-count outputs; NB07 §8. |
| **Table E.2 - Raw vs isotonic-calibrated benchmark coverage** | NB10 §6. |
| **Table E.3 - Prepayment-restricted reliability** | NB06 §8b. |
| **Table E.4 - Coarse Mondrian partitions** | NB05b §3, NB06 §§2–3, NB07 §7. |
| **Table E.5 - Coverage by calibrated-probability decile** | NB08 §2. |
| **Table E.6 - Miscoverage-model discrimination** | NB08 §3. |
| **Table E.7 - Miscoverage concentration** | NB08 §4. |
| **Table E.8 - Depth-limited reliability pockets** | NB08 §5. |
| **Table E.9 - Miscoverage by borrower-risk cell** | NB08 §2. |
| **Table E.10 - Adaptive coverage / long-run reliability** | NB06 §§5–6; presentation/synthesis in NB07 §9. |
| **Table E.11 - Static-method reliability under logistic benchmark** | NB10 §§4–5. |
| **Table F.1 - Score-distribution movement and PSI by window** | NB05a §3; NB06 §4; assembled/interpreted in NB07 §10. |
| **Table F.2 - A priori regime labels against realised evidence** | NB07 §10, using supporting diagnostics from NB02/NB03. |
| **Table F.3 - Feature-level stability indices** | NB02 §5.9. |
| **Table F.4 - Stability/score movement under logistic benchmark** | NB10 §§4.4, 5.4. |
| **Table F.5 - Monitoring-band classification vs reliability ordering** | NB05a §3; NB06 §§3–4; NB10 §§4.4–5.4. |
| **Table G.1 - Review budget and event rate by borrower-risk cell** | NB09 §7. |
| **Table G.2 - Monthly review budget by method and regime** | NB09 §8. |
| **Table G.3 - Absolute reviewed/captured volumes** | NB09 §4. |
| **Table G.4 - Matched-budget capture comparison** | NB09 §§5–6. |
| **Table G.5 - Matched-budget capture under logistic benchmark** | NB10 §7. |
| **Table G.6 - Escalation boundary / realised escalation rate** | NB09 §§1–2. |

---

## 7. Navigation by research question

| Research question | Start | Then inspect |
|---|---|---|
| **RQ1** | NB05b §§2, 7 | NB06 §§2–4, 6, 8b → NB07 §§4–6, 9–10 → NB08 §§2–6 → NB10 §§4–6 |
| **RQ2** | NB06 §§2–6 | NB06 §8b → NB07 §§4–9 → NB08 §§2–7 → NB10 §§4–5 |
| **RQ3** | NB05a §3 | NB05b §7 → NB06 §§4, 6 → NB07 §§9–10 → NB02 §5.9 → NB10 §§4–5 |
| **RQ4** | NB09 §§1–6 | NB09 §§7–8 → NB10 §7 |